# 逐行构造交易特征 · 原笔记本逐行精读版

本文件保留原 Notebook 的代码顺序、代码文本和原有输出，在每格代码前插入中文解析。原单元格编号指原文件 cells 数组的位置，从 1 开始，包含 Markdown 与空格；它不是执行计数 In[n]。

**本次未执行原代码。已有输出来自原书历史记录。手工构造的小例子会明确标注，不是运行原程序得到的结果。** 静态逐行层解释语法与执行顺序，专项补注解释关键算法、数据形状与可复核的小数据。若遇原书旧接口或实现缺点，会说明而不暗改原始代码。

原文件：`Chapter_3_GettingStarted/BaselineFeatureTransformation.ipynb`。原书代码 GPL v3；原文及图片 CC BY-SA 4.0。原作者：Yann-Aël Le Borgne、Wissam Siblini、Bertrand Lebichot、Gianluca Bontempi。中文补注为新增教学改编，遵循 CC BY-SA 4.0。

这是一份阅读用副本。运行代码可能下载数据、训练模型或写文件；阅读不要求执行。

(Baseline_Feature_Transformation)=
# Baseline feature transformation

The simulated dataset generated in the previous section is simple. It only contains the essential features that characterize a payment card transaction. These are: a unique identifier for the transaction, the date and time of the transaction, the transaction amount, a unique identifier for the customer, a unique number for the merchant, and a binary variable that labels the transaction as legitimate or fraudulent (0 for legitimate or 1 for fraudulent). Fig. 1 provides the first three rows of the simulated dataset:
 
![alt text](images/tx_table.png)
<p style="text-align: center;">
Fig. 1. The first three transactions in the simulated dataset used in this chapter.
</p>



What each row essentially summarizes is that, at 00:00:31, on the 1st of April 2018, a customer with ID 596 made a payment of a value of 57.19 to a merchant with ID 3156, and that the transaction was not fraudulent. Then, at 00:02:10, on the 1st of April 2018, a customer with ID 4961 made a payment of a value of 81.51 to a merchant with ID 3412, and that the transaction was not fraudulent. And so on. The simulated dataset is a long list of such transactions (1.8 million in total). The variable `transaction_ID` is a unique identifier for each transaction.  

While conceptually simple for a human, such a set of features is however not appropriate for a machine learning predictive model. Machine learning algorithms typically require *numerical* and *ordered* features. Numerical means that the type of the variable must be an integer or a real number. Ordered means that the order of the values of a variable is meaningful. 

In this dataset, the only numerical and ordered features are the transaction amount and the fraud label. The date is a Panda timestamp, and therefore not numerical. The identifiers for the transactions, customers, and terminals are numerical but not ordered: it would not make sense to assume for example that the terminal with ID 3548 is 'bigger' or 'larger' than the terminal with ID 1983. Rather, these identifiers represent distinct 'entities', which are referred to as *categorical* features. 

There is unfortunately no standard procedure to deal with non-numerical or categorical features. The topic is known in the machine learning literature as *feature engineering* or *feature transformation*. In essence, the goal of feature engineering is to design new features that are assumed to be relevant for a predictive problem. The design of these features is usually problem-dependent, and involves domain knowledge.

In this section, we will implement three types of feature transformation that are known to be relevant for payment card fraud detection.

![encoding](images/encoding_variables.png)

The first type of transformation involves the date/time variable, and consists in creating binary features that characterize potentially relevant periods. We will create two such features. The first one will characterize whether a transaction occurs during a weekday or during the weekend. The second will characterize whether a transaction occurs during the day or the night. These features can be useful since it has been observed in real-world datasets that fraudulent patterns differ between weekdays and weekends, and between the day and night.  

The second type of transformation involves the customer ID and consists in creating features that characterize the customer spending behaviors. We will follow the RFM (Recency, Frequency, Monetary value) framework proposed in {cite}`VANVLASSELAER201538`, and keep track of the average spending amount and number of transactions for each customer and for three window sizes. This will lead to the creation of six new features.

The third type of transformation involves the terminal ID and consists in creating new features that characterize the 'risk' associated with the terminal. The risk will be defined as the average number of frauds that were observed on the terminal for three window sizes. This will lead to the creation of three new features. 

The table below summarizes the types of transformation that will be performed and the new features that will be created. 

|Original feature name|Original feature type|Transformation|Number of new features|New feature(s) type|
|---|---|---|---|---|
|TX\_DATE\_TIME | Panda timestamp |0 if transaction during a weekday, 1 if transaction during a weekend. The new feature is called TX_DURING_WEEKEND.|1|Integer (0/1)|
|TX\_DATE\_TIME | Panda timestamp |0 if transaction between 6am and 0pm, 1 if transaction between 0pm and 6am. The new feature is called TX_DURING_NIGHT.|1|Integer (0/1)|
|CUSTOMER\_ID | Categorical variable |Number of transactions by the customer in the last n day(s), for n in {1,7,30}. The new features are called CUSTOMER_ID_NB_TX_nDAY_WINDOW.|3|Integer|
|CUSTOMER\_ID | Categorical variable |Average spending amount in the last n day(s), for n in {1,7,30}. The new features are called CUSTOMER_ID_AVG_AMOUNT_nDAY_WINDOW.|3|Real|
|TERMINAL\_ID | Categorical variable |Number of transactions on the terminal in the last n+d day(s), for n in {1,7,30} and d=7. The parameter d is called delay and will be discussed later in this notebook. The new features are called TERMINAL_ID_NB_TX_nDAY_WINDOW.|3|Integer|
|TERMINAL\_ID | Categorical variable |Average number of frauds on the terminal in the last n+d day(s), for n in {1,7,30} and d=7. The parameter d is called delay and will be discussed later in this notebook. The new features are called TERMINAL_ID_RISK_nDAY_WINDOW.|3|Real|

The following sections provide the implementation for each of these three transformations. After the transformations, a set of 14 new features will be created. Note that some of the features are the result of aggregation functions over the values of other features or conditions (same customer, given time window). These features are often referred to as *aggregated features*.

## 源码精读 · 原单元格 4

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 加载公共函数，并在需要时获取原始模拟数据。

**执行前：** Notebook 环境应能执行 IPython 命令；目录和依赖是否就绪不能只凭代码断定。

**执行后：** 实际执行后，公共函数和导入名称进入当前会话；满足条件时会发生下载和仓库克隆。此精读页面不会执行这些命令。

按顺序读三件事：`!curl -O ...` 让操作系统下载文件；`%run shared_functions.py` 执行这个文件；`if not os.path.exists(...)` 判断数据目录不存在时才进入克隆分支。`!` 和 `%` 都是 Notebook 提供的扩展语法，不能原样当作普通 Python 脚本语法。

`read_from_files`、模型库等名称可能来自公共文件，因此不能因为本页开头没写某条 `import` 就判断后面的名称必然不存在。反过来，如果前面的下载或执行失败，也不能假设这些名称已经可用。这格是准备工作，没有生成新的客户交易特征。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
# Initialization: Load shared functions and simulated data 
```

执行位置：本行是注释，不执行。所在上下文：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

原作者的说明性注释：Initialization: Load shared functions and simulated data

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。

**第 3 行**

```python
# Load shared functions
```

执行位置：本行是注释，不执行。所在上下文：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

原作者的说明性注释：Load shared functions

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。

**第 4 行**

```python
!curl -O https://raw.githubusercontent.com/Fraud-Detection-Handbook/fraud-detection-handbook/main/Chapter_References/shared_functions.py
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

这是一条 Notebook/IPython 扩展命令，不是普通 Python 表达式。以 ! 开头会调用系统命令；这里可能下载文件或克隆数据仓库，阅读本解析不会执行它。

只有你真正执行该单元格时才发生上述动作；本解析只说明原命令。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。

**第 5 行**

```python
%run shared_functions.py
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

这是一条 Notebook/IPython 扩展命令，不是普通 Python 表达式。%run 会执行外部脚本并引入其定义。

只有你真正执行该单元格时才发生上述动作；本解析只说明原命令。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。

**第 7 行**

```python
# Get simulated data from Github repository
```

执行位置：本行是注释，不执行。所在上下文：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

原作者的说明性注释：Get simulated data from Github repository

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。

**第 8 行**

```python
if not os.path.exists("simulated-data-raw"):
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

判断条件：对条件取逻辑否定，作用对象是 `os.path.exists('simulated-data-raw')`。

条件成立才执行相应缩进分支，不成立则转到后续 elif/else 或跳过。

- 求值 `os.path.exists('simulated-data-raw')`：检查路径是否存在，返回布尔值。传入：位置参数1是 `'simulated-data-raw'`
- 求值 `not os.path.exists('simulated-data-raw')`：对条件取逻辑否定，作用对象是 `os.path.exists('simulated-data-raw')`

- 调用 `os.path.exists` 时的位置参数1：`'simulated-data-raw'`；先求出该表达式再传入

- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。

**第 9 行**

```python
    !git clone https://github.com/Fraud-Detection-Handbook/simulated-data-raw
```

执行位置：仅在条件 `not os.path.exists('simulated-data-raw')` 成立的分支中执行

这是一条 Notebook/IPython 扩展命令，不是普通 Python 表达式。以 ! 开头会调用系统命令；这里可能下载文件或克隆数据仓库，阅读本解析不会执行它。

只有你真正执行该单元格时才发生上述动作；本解析只说明原命令。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [24]:
# Initialization: Load shared functions and simulated data 

# Load shared functions
!curl -O https://raw.githubusercontent.com/Fraud-Detection-Handbook/fraud-detection-handbook/main/Chapter_References/shared_functions.py
%run shared_functions.py

# Get simulated data from Github repository
if not os.path.exists("simulated-data-raw"):
    !git clone https://github.com/Fraud-Detection-Handbook/simulated-data-raw
        

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31567  100 31567    0     0   135k      0 --:--:-- --:--:-- --:--:--  135k
Cloning into 'simulated-data-raw'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 189 (delta 0), reused 186 (delta 0), pack-reused 0
Receiving objects: 100% (189/189), 28.04 MiB | 3.13 MiB/s, done.


## Loading of dataset

Let us first load the transaction data simulated in the previous notebook. We will load the transaction files from April to September. Files can be loaded using the `read_from_files` function in the [shared functions](shared_functions) notebook. The function was put in this notebook since it will be used frequently throughout this book.

The function takes as input the folder where the data files are located, and the dates that define the period to load (between `BEGIN_DATE` and `END_DATE`). It returns a DataFrame of transactions. The transactions are sorted by chronological order. 


## 源码精读 · 原单元格 6

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 把指定日期范围内保存的交易文件读成一张交易表。

**执行前：** 已有 read_from_files 函数，目录下应有按日期命名的数据文件。

**执行后：** transactions_df 引用读取结果；屏幕打印交易笔数和 TX_FRAUD 求和得到的欺诈笔数。

`DIR_INPUT` 是路径字符串，`BEGIN_DATE` 和 `END_DATE` 也是字符串；它们此时不是时间对象。真正如何筛选文件由 `read_from_files` 的函数体决定，公共函数索引可以继续追进去。

`%time transactions_df=...` 先调用读取函数，再把返回表绑定给变量，同时显示耗时。`len(transactions_df)` 数表的行数；`transactions_df.TX_FRAUD.sum()` 将 0/1 标签相加，得到欺诈**交易**数，并不是欺诈客户数。假设人工小表标签是 `[0,1,1,0]`，这里显示总数 4、欺诈数 2；若两笔欺诈属于同一客户，受影响客户仍只有 1。这个小例子不是原书实际读取结果。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
DIR_INPUT='./simulated-data-raw/data/' 
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：字符串 `./simulated-data-raw/data/`（文字，不是同名变量）。

把结果绑定到 `DIR_INPUT`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。

**第 3 行**

```python
BEGIN_DATE = "2018-04-01"
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：字符串 `2018-04-01`（文字，不是同名变量）。

把结果绑定到 `BEGIN_DATE`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。

**第 4 行**

```python
END_DATE = "2018-09-30"
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：字符串 `2018-09-30`（文字，不是同名变量）。

把结果绑定到 `END_DATE`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。

**第 6 行**

```python
print("Load  files")
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

把结果显示出来；显示本身不等于将结果交给另一个变量。传入：位置参数1是 `'Load  files'`。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `print('Load  files')`：把结果显示出来；显示本身不等于将结果交给另一个变量。传入：位置参数1是 `'Load  files'`

- 调用 `print` 时的位置参数1：`'Load  files'`；先求出该表达式再传入

- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。

**第 7 行**

```python
%time transactions_df=read_from_files(DIR_INPUT, BEGIN_DATE, END_DATE)
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

IPython 的 %time 会执行后面的语句并记录耗时；计时结果取决于运行环境。 先计算右侧：调用 `read_from_files`；它的返回值与副作用由此函数或类的定义决定，不能只凭名字推定数值。传入：位置参数1是 `DIR_INPUT`；位置参数2是 `BEGIN_DATE`；位置参数3是 `END_DATE`。

把结果绑定到 `transactions_df`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `read_from_files(DIR_INPUT, BEGIN_DATE, END_DATE)`：调用 `read_from_files`；它的返回值与副作用由此函数或类的定义决定，不能只凭名字推定数值。传入：位置参数1是 `DIR_INPUT`；位置参数2是 `BEGIN_DATE`；位置参数3是 `END_DATE`

- 调用 `read_from_files` 时的位置参数1：`DIR_INPUT`；先求出该表达式再传入
- 调用 `read_from_files` 时的位置参数2：`BEGIN_DATE`；先求出该表达式再传入
- 调用 `read_from_files` 时的位置参数3：`END_DATE`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

**第 8 行**

```python
print("{0} transactions loaded, containing {1} fraudulent transactions".format(len(transactions_df),transactions_df.TX_FRAUD.sum()))
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

把结果显示出来；显示本身不等于将结果交给另一个变量。传入：位置参数1是 `'{0} transactions loaded, containing {1} fraudulent transactions'.format(len(transactions_df), transactions_df.TX_FRAUD.sum())`。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `len(transactions_df)`：取得对象长度：列表是元素数，DataFrame通常是行数。传入：位置参数1是 `transactions_df`
- 求值 `transactions_df.TX_FRAUD.sum()`：对指定数据或窗口求和；0/1标签求和可统计欺诈笔数。这里没有显式传入参数
- 求值 `'{0} transactions loaded, containing {1} fraudulent transactions'.format(len(transactions_df), transactions_df.TX_FRAUD.sum())`：把给定值填入格式字符串的占位位置。传入：位置参数1是 `len(transactions_df)`；位置参数2是 `transactions_df.TX_FRAUD.sum()`
- 求值 `print('{0} transactions loaded, containing {1} fraudulent transactions'.format(len(transactions_df), transactions_df.TX_FRAUD.sum()))`：把结果显示出来；显示本身不等于将结果交给另一个变量。传入：位置参数1是 `'{0} transactions loaded, containing {1} fraudulent transactions'.format(len(transactions_df), transactions_df.TX_FRAUD.sum())`

- 调用 `print` 时的位置参数1：`'{0} transactions loaded, containing {1} fraudulent transactions'.format(len(transactions_df), transactions_df.TX_FRAUD.sum())`；先求出该表达式再传入
- 调用 `'{0} transactions loaded, containing {1} fraudulent transactions'.format` 时的位置参数1：`len(transactions_df)`；先求出该表达式再传入
- 调用 `'{0} transactions loaded, containing {1} fraudulent transactions'.format` 时的位置参数2：`transactions_df.TX_FRAUD.sum()`；先求出该表达式再传入
- 调用 `len` 时的位置参数1：`transactions_df`；先求出该表达式再传入

- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

- `TX_FRAUD`：0/1 欺诈标签，属于监督目标或事后评估信息
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [3]:
DIR_INPUT='./simulated-data-raw/data/' 

BEGIN_DATE = "2018-04-01"
END_DATE = "2018-09-30"

print("Load  files")
%time transactions_df=read_from_files(DIR_INPUT, BEGIN_DATE, END_DATE)
print("{0} transactions loaded, containing {1} fraudulent transactions".format(len(transactions_df),transactions_df.TX_FRAUD.sum()))


Load  files
CPU times: user 3.1 s, sys: 696 ms, total: 3.79 s
Wall time: 4.13 s
1754155 transactions loaded, containing 14681 fraudulent transactions


## 源码精读 · 原单元格 7

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 查看读取结果前几行的列与值。

**执行前：** transactions_df 已存在。

**执行后：** head() 返回前五行或不足五行的全部行供展示；原表不被缩短。

把 `transactions_df` 看成一本完整账册，`.head()` 是打开前几条记录。它没有把变量重新赋值，也没有删除后续交易。阅读输出时先认列名、索引和单元格类型；不要把左侧索引自动当成 `TRANSACTION_ID`。前五行只是样本，不能据此证明整个半年没有缺失值或异常值。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
transactions_df.head()
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

取前 n 行，默认前五行；用于查看不等于检查了全表。这里没有显式传入参数。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `transactions_df.head()`：取前 n 行，默认前五行；用于查看不等于检查了全表。这里没有显式传入参数

- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。

- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [4]:
transactions_df.head()

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0


## Date and time transformations

We will create two new binary features from the transaction dates and times:

* The first will characterize whether a transaction occurs during a weekday (value 0) or a weekend (1), and will be called `TX_DURING_WEEKEND`
* The second will characterize whether a transaction occurs during the day or during the day (0) or during the night (1). The night is defined as hours that are between 0pm and 6am. It will be called `TX_DURING_NIGHT`. 

For the `TX_DURING_WEEKEND` feature, we define a function `is_weekend` that takes as input a Panda timestamp, and returns 1 if the date is during a weekend, or 0 otherwise. The timestamp object conveniently provides the `weekday` function to help in computing this value.

## 源码精读 · 原单元格 9

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 定义将单个时间转换为周末标记的函数。

**执行前：** 输入约定是一个支持 weekday() 的日期时间对象。

**执行后：** 执行 def 只登记 is_weekend；调用后才返回整数 0 或 1。

人工跟踪：传入星期日的某个时间 → `weekday()` 返回 6 → `weekday>=5` 为 `True` → 局部变量 `is_weekend` 保存布尔值 True → `int(is_weekend)` 转成整数 1 → `return` 把 1 交还调用者。换成星期一，星期编号为 0，比较结果 False 转成整数 0。

这份原代码没有写 if/else，而是直接保存比较结果，再用 int 转换。函数外的 `is_weekend` 是函数名称，函数内同名局部变量装布尔值；这个命名虽能运行，但初学时很容易混淆。函数体里的数值不会自动成为整张表的一列，下一次 `.apply(...)` 才逐个调用它。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
def is_weekend(tx_datetime):
```

执行位置：此处是函数 `is_weekend` 的定义签名；默认值表达式在到达定义时求值，函数体等待以后调用

定义函数 `is_weekend(tx_datetime)`。冒号后缩进的代码是它的函数体。

到达此定义时求值默认参数等定义期表达式，建立函数对象并绑定名称；不会因此执行函数体。函数体说明仅在以后调用并走到相应路径时成立。

- 定义参数 `tx_datetime`，本签名未给默认值，调用者需按参数规则提供

- def 后的外层圆括号列出定义参数，不是调用；其中默认表达式若含另一个函数调用，它仍在定义时求值。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。

**第 3 行**

```python
    # Transform date into weekday (0 is Monday, 6 is Sunday)
```

执行位置：本行是注释，不执行。所在上下文：此处是函数 `is_weekend` 的定义签名；默认值表达式在到达定义时求值，函数体等待以后调用

原作者的说明性注释：Transform date into weekday (0 is Monday, 6 is Sunday)

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 4 行**

```python
    weekday = tx_datetime.weekday()
```

执行位置：仅在调用 `is_weekend` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：返回星期编号，周一0、周日6。这里没有显式传入参数。

把结果绑定到 `weekday`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `tx_datetime.weekday()`：返回星期编号，周一0、周日6。这里没有显式传入参数

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 5 行**

```python
    # Binary value: 0 if weekday, 1 if weekend
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `is_weekend` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：Binary value: 0 if weekday, 1 if weekend

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 6 行**

```python
    is_weekend = weekday>=5
```

执行位置：仅在调用 `is_weekend` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：判断`weekday` 是否大于或等于 `5`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值。

把结果绑定到 `is_weekend`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `weekday >= 5`：判断`weekday` 是否大于或等于 `5`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 8 行**

```python
    return int(is_weekend)
```

执行位置：仅在调用 `is_weekend` 并执行到本行时发生；仅执行 def 不执行这段函数体

计算要返回的内容：把输入转换为整数；浮点数转换时向零截去小数部分，不是取最接近的整数。传入：位置参数1是 `is_weekend`。

立即结束本次函数调用，把这个结果交回调用处；这次函数调用中后面的语句不再执行。返回表不等于自动写入磁盘。

- 求值 `int(is_weekend)`：把输入转换为整数；浮点数转换时向零截去小数部分，不是取最接近的整数。传入：位置参数1是 `is_weekend`

- 调用 `int` 时的位置参数1：`is_weekend`；先求出该表达式再传入

- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [5]:
def is_weekend(tx_datetime):
    
    # Transform date into weekday (0 is Monday, 6 is Sunday)
    weekday = tx_datetime.weekday()
    # Binary value: 0 if weekday, 1 if weekend
    is_weekend = weekday>=5
    
    return int(is_weekend)


It is then straghtforward to compute this feature for all transactions using the Panda `apply` function. 

## 源码精读 · 原单元格 11

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 把周末函数应用到每笔交易，并写入新列。

**执行前：** TX_DATETIME 是长度 N 的时间列；is_weekend 是已定义函数。

**执行后：** 原表新增或覆盖 TX_DURING_WEEKEND；正常情况下行数仍为 N。

由内向外读：`transactions_df.TX_DATETIME` 取出一列 Series；`.apply(is_weekend)` 把每个时间逐个交给函数；右侧生成与这些行对应的 0/1 Series；左侧方括号指定写入目标列。

人工三行是周五、周六、周日，右侧结果就是 `[0,1,1]`。左边每一行原来的金额、客户编号仍在，新列把结果贴回相应行。`apply(is_weekend)` 传的是函数对象；写成 `apply(is_weekend())` 会先无参数调用函数，不符合这里需要的用法。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
%time transactions_df['TX_DURING_WEEKEND']=transactions_df.TX_DATETIME.apply(is_weekend)
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

IPython 的 %time 会执行后面的语句并记录耗时；计时结果取决于运行环境。 先计算右侧：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `is_weekend`。

再把结果写到 `transactions_df['TX_DURING_WEEKEND']` 指定的位置或列；若是DataFrame列赋值，需要检查行数、索引对齐和是否覆盖已有列。

- 求值 `transactions_df.TX_DATETIME.apply(is_weekend)`：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `is_weekend`

- 调用 `transactions_df.TX_DATETIME.apply` 时的位置参数1：`is_weekend`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。

- `TX_DATETIME`：交易发生的日期时间
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [6]:
%time transactions_df['TX_DURING_WEEKEND']=transactions_df.TX_DATETIME.apply(is_weekend)

CPU times: user 7.54 s, sys: 247 ms, total: 7.79 s
Wall time: 7.94 s


We follow the same logic to implement the `TX_DURING_NIGHT` feature. First, a function `is_night` that takes as input a Panda timestamp, and returns 1 if the time is during the night, or 0 otherwise. The timestamp object conveniently provides the hour property to help in computing this value.

## 源码精读 · 原单元格 13

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 定义夜间交易标记。

**执行前：** 输入是单个日期时间对象。

**执行后：** 登记 is_night；实际调用返回 0 或 1。

`.hour` 是属性，取整数小时，不带调用圆括号。人工例子：05:30 的小时是 5，06:59 的小时是 6，07:00 的小时是 7。按原代码 `tx_hour<=6`，前两笔都得到 1，最后一笔得到 0。

这里应以实际比较条件为准：注释说 less than 6，但代码使用小于或等于 6，因此覆盖了 06:00 到 06:59。不要将这条笔记本规则背成普遍的银行夜间业务定义；它只是作者为此示例选定的特征口径。先知道代码到底做了什么，再决定自己的项目要采用哪一个时间边界。

**需要辨清的地方**

- 原注释与 <=6 的边界不完全一致。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
def is_night(tx_datetime):
```

执行位置：此处是函数 `is_night` 的定义签名；默认值表达式在到达定义时求值，函数体等待以后调用

定义函数 `is_night(tx_datetime)`。冒号后缩进的代码是它的函数体。

到达此定义时求值默认参数等定义期表达式，建立函数对象并绑定名称；不会因此执行函数体。函数体说明仅在以后调用并走到相应路径时成立。

- 定义参数 `tx_datetime`，本签名未给默认值，调用者需按参数规则提供

- def 后的外层圆括号列出定义参数，不是调用；其中默认表达式若含另一个函数调用，它仍在定义时求值。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。

**第 3 行**

```python
    # Get the hour of the transaction
```

执行位置：本行是注释，不执行。所在上下文：此处是函数 `is_night` 的定义签名；默认值表达式在到达定义时求值，函数体等待以后调用

原作者的说明性注释：Get the hour of the transaction

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 4 行**

```python
    tx_hour = tx_datetime.hour
```

执行位置：仅在调用 `is_night` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：访问 `tx_datetime` 的 `hour`，取得时间对象的小时部分。

把结果绑定到 `tx_hour`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 5 行**

```python
    # Binary value: 1 if hour less than 6, and 0 otherwise
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `is_night` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：Binary value: 1 if hour less than 6, and 0 otherwise

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 6 行**

```python
    is_night = tx_hour<=6
```

执行位置：仅在调用 `is_night` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：判断`tx_hour` 是否小于或等于 `6`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值。

把结果绑定到 `is_night`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `tx_hour <= 6`：判断`tx_hour` 是否小于或等于 `6`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 8 行**

```python
    return int(is_night)
```

执行位置：仅在调用 `is_night` 并执行到本行时发生；仅执行 def 不执行这段函数体

计算要返回的内容：把输入转换为整数；浮点数转换时向零截去小数部分，不是取最接近的整数。传入：位置参数1是 `is_night`。

立即结束本次函数调用，把这个结果交回调用处；这次函数调用中后面的语句不再执行。返回表不等于自动写入磁盘。

- 求值 `int(is_night)`：把输入转换为整数；浮点数转换时向零截去小数部分，不是取最接近的整数。传入：位置参数1是 `is_night`

- 调用 `int` 时的位置参数1：`is_night`；先求出该表达式再传入

- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [7]:
def is_night(tx_datetime):
    
    # Get the hour of the transaction
    tx_hour = tx_datetime.hour
    # Binary value: 1 if hour less than 6, and 0 otherwise
    is_night = tx_hour<=6
    
    return int(is_night)

## 源码精读 · 原单元格 14

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 给每笔交易写入夜间标记。

**执行前：** 存在 TX_DATETIME 和 is_night。

**执行后：** 新增 TX_DURING_NIGHT，行数不因这次赋值改变。

执行路线与周末标记相同，但判断依据换成小时。人工三行时间 00:10、06:59、16:20，结果分别为 1、1、0。`.apply` 没有把整张表一次传给 `is_night`，这里接收对象是一列，所以函数看到的是一个个时间值。

先看等号右侧的结果，再看左侧列名，能避免误把 `TX_DURING_NIGHT` 当成函数参数。计时输出仅描述这次环境中的耗时，不能直接用原书保存的秒数预测你的电脑。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
%time transactions_df['TX_DURING_NIGHT']=transactions_df.TX_DATETIME.apply(is_night)
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

IPython 的 %time 会执行后面的语句并记录耗时；计时结果取决于运行环境。 先计算右侧：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `is_night`。

再把结果写到 `transactions_df['TX_DURING_NIGHT']` 指定的位置或列；若是DataFrame列赋值，需要检查行数、索引对齐和是否覆盖已有列。

- 求值 `transactions_df.TX_DATETIME.apply(is_night)`：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `is_night`

- 调用 `transactions_df.TX_DATETIME.apply` 时的位置参数1：`is_night`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。

- `TX_DATETIME`：交易发生的日期时间
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [8]:
%time transactions_df['TX_DURING_NIGHT']=transactions_df.TX_DATETIME.apply(is_night)

CPU times: user 7.09 s, sys: 221 ms, total: 7.31 s
Wall time: 7.47 s


Let us check that these features where correctly computed.

## 源码精读 · 原单元格 16

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 显示第 30 天及以后的交易子集。

**执行前：** TX_TIME_DAYS 是相对起点的天编号。

**执行后：** 表达式产生筛选结果供展示；本格没有重新赋给 transactions_df。

`transactions_df.TX_TIME_DAYS>=30` 先得到布尔条件。人工天编号 `[0,29,30,31]` 对应 `[False,False,True,True]`；外层 `transactions_df[条件]` 取出后两行。这个子集是表达式的结果，不意味着原变量已经只剩两行。

这里的 30 是日编号阈值，不是 30 行。若一天有很多笔交易，它们会一起按各自的天编号判断。是否保留这个筛选结果用于后续训练，要看后面是否有赋值或在其他函数中重新筛选，不能仅凭这一格展示推断。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
transactions_df[transactions_df.TX_TIME_DAYS>=30]
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

从 `transactions_df` 中按 `transactions_df.TX_TIME_DAYS >= 30` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `transactions_df.TX_TIME_DAYS >= 30`：判断`transactions_df.TX_TIME_DAYS` 是否大于或等于 `30`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值
- 求值 `transactions_df[transactions_df.TX_TIME_DAYS >= 30]`：从 `transactions_df` 中按 `transactions_df.TX_TIME_DAYS >= 30` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则

- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。

- `TX_TIME_DAYS`：相对模拟起点的整天编号
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [9]:
transactions_df[transactions_df.TX_TIME_DAYS>=30]

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT
288062,288062,2018-05-01 00:01:21,3546,2944,18.71,2592081,30,0,0,0,1
288063,288063,2018-05-01 00:01:48,206,3521,18.60,2592108,30,0,0,0,1
288064,288064,2018-05-01 00:02:22,2610,4470,66.67,2592142,30,0,0,0,1
288065,288065,2018-05-01 00:03:15,4578,1520,79.41,2592195,30,0,0,0,1
288066,288066,2018-05-01 00:03:51,1246,7809,52.08,2592231,30,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
1754150,1754150,2018-09-30 23:56:36,161,655,54.24,15810996,182,0,0,1,0
1754151,1754151,2018-09-30 23:57:38,4342,6181,1.23,15811058,182,0,0,1,0
1754152,1754152,2018-09-30 23:58:21,618,1502,6.62,15811101,182,0,0,1,0
1754153,1754153,2018-09-30 23:59:52,4056,3067,55.40,15811192,182,0,0,1,0


The 2018-05-01 was a Monday, and the 2018-09-30 a Sunday. These dates are correctly flagged as weekday, and weekend, respectively. The day and night feature is also correctly set for the first transactions, that happen closely after 0 pm, and the last transactions that happen closely before 0 pm. 

## Customer ID transformations

Let us now proceed with customer ID transformations. We will take inspiration from the RFM (Recency, Frequency, Monetary value) framework proposed in {cite}`VANVLASSELAER201538`, and compute two of these features over three time windows. The first feature will be the number of transactions that occur within a time window (Frequency). The second will be the average amount spent in these transactions (Monetary value). The time windows will be set to one, seven, and thirty days. This will generate six new features. Note that these time windows could later be optimized along with the models using a model selection procedure ([Chapter 5](Model_Selection)). 

Let us implement these transformations by writing a `get_customer_spending_behaviour_features` function. The function takes as inputs the set of transactions for a customer and a set of window sizes. It returns a DataFrame with the six new features. Our implementation relies on the Panda `rolling` function, which makes easy the computation of aggregates over a time window.


## 源码精读 · 原单元格 19

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 定义一个客户的消费次数与平均金额滚动特征。

**执行前：** 输入是一位客户的交易表，至少有 TX_DATETIME、TX_AMOUNT 和 TRANSACTION_ID。

**执行后：** 定义阶段不计算；调用时返回同一批交易加六列：1、7、30 天的笔数与平均金额。

先按时间排序，然后把行索引临时设成 `TX_DATETIME`。这样 `.rolling('1d')` 的窗口是一天时间长度，不是一行；`str(window_size)+'d'` 把整数 1 变成字符串 `'1d'`。

人工小表：客户 A 在 4 月 1 日 10:00 消费 20，4 月 1 日 18:00 消费 40，4 月 2 日 12:00 消费 100。看第三行的一天窗口 `(4月1日12:00, 4月2日12:00]`，只包含 40 和 100，`sum=140`、`count=2`、`avg=70`。第三行七天窗口包含三笔，`sum=160`、`count=3`、`avg=160/3`。这些数是人为设计的，用来显示窗口边界。

循环第一次令 `window_size=1`，生成一对列；第二次为 7，第三次为 30。`NB_TX_WINDOW` 是中间 Series，`AVG_AMOUNT_TX_WINDOW=SUM_AMOUNT_TX_WINDOW/NB_TX_WINDOW` 按相同行标签对应相除。随后 `list(...)` 取出顺序值，赋回等长的表列；这里转换为列表意味着按位置写入，不能再依靠原 Series 索引自动纠正乱序。

末尾把索引改成 `TRANSACTION_ID`，然后 `return customer_transactions` 返回整张加工后的表。赋值给 `.index` 与删除对应数据列不同，交易 ID 列仍可留在表里。由于默认时间窗口包含当前交易，第三行的 100 已经参加该行平均值；这适合交易发生时金额已经可知的评分场景，但不能把它描述为完全排除本笔的纯历史均值。

函数默认参数 `[1,7,30]` 在定义时创建，原函数只遍历它而没有追加元素；若将来改写为原地修改这个列表，应另考虑可变默认值问题。

**需要辨清的地方**

- 先确保按客户分别调用，不能把不同客户混到同一窗口。
- 时间重复、缺失金额和索引对齐需要额外检查；此例的简化数据不能代表所有生产输入。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
def get_customer_spending_behaviour_features(customer_transactions, windows_size_in_days=[1,7,30]):
```

执行位置：此处是函数 `get_customer_spending_behaviour_features` 的定义签名；默认值表达式在到达定义时求值，函数体等待以后调用

定义函数 `get_customer_spending_behaviour_features(customer_transactions, windows_size_in_days=[1, 7, 30])`。冒号后缩进的代码是它的函数体。

到达此定义时求值默认参数等定义期表达式，建立函数对象并绑定名称；不会因此执行函数体。函数体说明仅在以后调用并走到相应路径时成立。

- 求值 `[1, 7, 30]`：含 3 项的列表：常量 `1`；常量 `7`；常量 `30`

- 定义参数 `customer_transactions`，本签名未给默认值，调用者需按参数规则提供
- 定义参数 `windows_size_in_days` 的默认表达式 `[1, 7, 30]`；它在定义时求值一次，调用未提供该参数时才采用这个默认对象

- 函数定义签名中的 `参数名=表达式` 指定默认值；默认表达式在定义时求值，不能当成一次实际传参或函数体赋值。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- def 后的外层圆括号列出定义参数，不是调用；其中默认表达式若含另一个函数调用，它仍在定义时求值。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

- `windows_size_in_days`：需要分别计算的多个窗口天数

**第 3 行**

```python
    # Let us first order transactions chronologically
```

执行位置：本行是注释，不执行。所在上下文：此处是函数 `get_customer_spending_behaviour_features` 的定义签名；默认值表达式在到达定义时求值，函数体等待以后调用

原作者的说明性注释：Let us first order transactions chronologically

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 4 行**

```python
    customer_transactions=customer_transactions.sort_values('TX_DATETIME')
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_DATETIME'`。

把结果绑定到 `customer_transactions`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `customer_transactions.sort_values('TX_DATETIME')`：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_DATETIME'`

- 调用 `customer_transactions.sort_values` 时的位置参数1：`'TX_DATETIME'`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 6 行**

```python
    # The transaction date and time is set as the index, which will allow the use of the rolling function 
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：The transaction date and time is set as the index, which will allow the use of the rolling function

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 7 行**

```python
    customer_transactions.index=customer_transactions.TX_DATETIME
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：访问 `customer_transactions` 的 `TX_DATETIME` 属性或成员。

把结果保存到 `customer_transactions.index` 指定的对象属性；这会改变该对象的状态。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `TX_DATETIME`：交易发生的日期时间

**第 9 行**

```python
    # For each window size
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：For each window size

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 10 行**

```python
    for window_size in windows_size_in_days:
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

遍历 `windows_size_in_days`，每次把取出的项交给 `window_size`。

对每项执行缩进体；循环变量通常被下一项覆盖，累计结果需另放列表、表格或计数器。空可迭代对象不会执行循环体。

- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度
- `windows_size_in_days`：需要分别计算的多个窗口天数

**第 12 行**

```python
        # Compute the sum of the transaction amounts and the number of transactions for the given window size
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：Compute the sum of the transaction amounts and the number of transactions for the given window size

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 13 行**

```python
        SUM_AMOUNT_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').sum()
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：对指定数据或窗口求和；0/1标签求和可统计欺诈笔数。这里没有显式传入参数。

把结果绑定到 `SUM_AMOUNT_TX_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `customer_transactions['TX_AMOUNT']`：从 `customer_transactions` 中按 `'TX_AMOUNT'` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `str(window_size)`：按 `str` 对输入进行构造或类型转换。传入：位置参数1是 `window_size`
- 求值 `str(window_size) + 'd'`：先计算 `str(window_size)` 与 `'d'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `customer_transactions['TX_AMOUNT'].rolling(str(window_size) + 'd')`：建立滚动窗口视图，窗口按行数还是时间取决于参数；后面的 sum/count 等才计算值。传入：位置参数1是 `str(window_size) + 'd'`
- 求值 `customer_transactions['TX_AMOUNT'].rolling(str(window_size) + 'd').sum()`：对指定数据或窗口求和；0/1标签求和可统计欺诈笔数。这里没有显式传入参数

- 调用 `customer_transactions['TX_AMOUNT'].rolling` 时的位置参数1：`str(window_size) + 'd'`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度

**第 14 行**

```python
        NB_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').count()
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：统计非缺失记录数量；不要自动等同包含缺失的所有行数。这里没有显式传入参数。

把结果绑定到 `NB_TX_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `customer_transactions['TX_AMOUNT']`：从 `customer_transactions` 中按 `'TX_AMOUNT'` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `str(window_size)`：按 `str` 对输入进行构造或类型转换。传入：位置参数1是 `window_size`
- 求值 `str(window_size) + 'd'`：先计算 `str(window_size)` 与 `'d'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `customer_transactions['TX_AMOUNT'].rolling(str(window_size) + 'd')`：建立滚动窗口视图，窗口按行数还是时间取决于参数；后面的 sum/count 等才计算值。传入：位置参数1是 `str(window_size) + 'd'`
- 求值 `customer_transactions['TX_AMOUNT'].rolling(str(window_size) + 'd').count()`：统计非缺失记录数量；不要自动等同包含缺失的所有行数。这里没有显式传入参数

- 调用 `customer_transactions['TX_AMOUNT'].rolling` 时的位置参数1：`str(window_size) + 'd'`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度

**第 16 行**

```python
        # Compute the average transaction amount for the given window size
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

原作者的说明性注释：Compute the average transaction amount for the given window size

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 17 行**

```python
        # NB_TX_WINDOW is always >0 since current transaction is always included
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

原作者的说明性注释：NB_TX_WINDOW is always >0 since current transaction is always included

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 18 行**

```python
        AVG_AMOUNT_TX_WINDOW=SUM_AMOUNT_TX_WINDOW/NB_TX_WINDOW
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：先取得 变量 `SUM_AMOUNT_TX_WINDOW` 当前引用的值，再与 变量 `NB_TX_WINDOW` 当前引用的值 做“除以”；数组或Series时通常逐元素计算，形状或索引须兼容。

把结果绑定到 `AVG_AMOUNT_TX_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `SUM_AMOUNT_TX_WINDOW / NB_TX_WINDOW`：先取得 变量 `SUM_AMOUNT_TX_WINDOW` 当前引用的值，再与 变量 `NB_TX_WINDOW` 当前引用的值 做“除以”；数组或Series时通常逐元素计算，形状或索引须兼容

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 20 行**

```python
        # Save feature values
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

原作者的说明性注释：Save feature values

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 21 行**

```python
        customer_transactions['CUSTOMER_ID_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `NB_TX_WINDOW`。

再把结果写到 `customer_transactions['CUSTOMER_ID_NB_TX_' + str(window_size) + 'DAY_WINDOW']` 指定的位置或列；若是DataFrame列赋值，需要检查行数、索引对齐和是否覆盖已有列。

- 求值 `list(NB_TX_WINDOW)`：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `NB_TX_WINDOW`

- 调用 `list` 时的位置参数1：`NB_TX_WINDOW`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度

**第 22 行**

```python
        customer_transactions['CUSTOMER_ID_AVG_AMOUNT_'+str(window_size)+'DAY_WINDOW']=list(AVG_AMOUNT_TX_WINDOW)
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `AVG_AMOUNT_TX_WINDOW`。

再把结果写到 `customer_transactions['CUSTOMER_ID_AVG_AMOUNT_' + str(window_size) + 'DAY_WINDOW']` 指定的位置或列；若是DataFrame列赋值，需要检查行数、索引对齐和是否覆盖已有列。

- 求值 `list(AVG_AMOUNT_TX_WINDOW)`：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `AVG_AMOUNT_TX_WINDOW`

- 调用 `list` 时的位置参数1：`AVG_AMOUNT_TX_WINDOW`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度

**第 24 行**

```python
    # Reindex according to transaction IDs
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：Reindex according to transaction IDs

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 25 行**

```python
    customer_transactions.index=customer_transactions.TRANSACTION_ID
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：访问 `customer_transactions` 的 `TRANSACTION_ID` 属性或成员。

把结果保存到 `customer_transactions.index` 指定的对象属性；这会改变该对象的状态。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `TRANSACTION_ID`：交易标识

**第 27 行**

```python
    # And return the dataframe with the new features
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：And return the dataframe with the new features

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 28 行**

```python
    return customer_transactions
```

执行位置：仅在调用 `get_customer_spending_behaviour_features` 并执行到本行时发生；仅执行 def 不执行这段函数体

计算要返回的内容：变量 `customer_transactions` 当前引用的值。

立即结束本次函数调用，把这个结果交回调用处；这次函数调用中后面的语句不再执行。返回表不等于自动写入磁盘。

- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [10]:
def get_customer_spending_behaviour_features(customer_transactions, windows_size_in_days=[1,7,30]):
    
    # Let us first order transactions chronologically
    customer_transactions=customer_transactions.sort_values('TX_DATETIME')
    
    # The transaction date and time is set as the index, which will allow the use of the rolling function 
    customer_transactions.index=customer_transactions.TX_DATETIME
    
    # For each window size
    for window_size in windows_size_in_days:
        
        # Compute the sum of the transaction amounts and the number of transactions for the given window size
        SUM_AMOUNT_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').sum()
        NB_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').count()
    
        # Compute the average transaction amount for the given window size
        # NB_TX_WINDOW is always >0 since current transaction is always included
        AVG_AMOUNT_TX_WINDOW=SUM_AMOUNT_TX_WINDOW/NB_TX_WINDOW
    
        # Save feature values
        customer_transactions['CUSTOMER_ID_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
        customer_transactions['CUSTOMER_ID_AVG_AMOUNT_'+str(window_size)+'DAY_WINDOW']=list(AVG_AMOUNT_TX_WINDOW)
    
    # Reindex according to transaction IDs
    customer_transactions.index=customer_transactions.TRANSACTION_ID
        
    # And return the dataframe with the new features
    return customer_transactions


Let us compute these aggregates for the first customer.

## 源码精读 · 原单元格 21

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 先对客户 0 单独试算，检查函数返回的新列。

**执行前：** 整张表含许多客户；滚动函数已经定义。

**执行后：** spending_behaviour_customer_0 保存筛出的客户 0 的加工结果；本格没有把结果统一合并回全体客户。

最内层比较 `CUSTOMER_ID==0` 得到掩码，筛出该客户全部记录，然后整张客户子表作为函数的第一个参数。没有传第二个参数，因此采用默认 `[1,7,30]`。返回值由左边长变量名接住，下一行单独写变量让 Notebook 展示它。

人工例子：全表有 A 两行、B 三行，筛选 A 后函数只看那两行，不会将 B 的三笔算入 A 的历史。这个试算的价值是检查一名客户按时间相邻的几行，亲手验证新列，而非只看全表出现了几个新字段。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
spending_behaviour_customer_0=get_customer_spending_behaviour_features(transactions_df[transactions_df.CUSTOMER_ID==0])
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：调用 `get_customer_spending_behaviour_features`；它的返回值与副作用由此函数或类的定义决定，不能只凭名字推定数值。传入：位置参数1是 `transactions_df[transactions_df.CUSTOMER_ID == 0]`。

把结果绑定到 `spending_behaviour_customer_0`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `transactions_df.CUSTOMER_ID == 0`：判断`transactions_df.CUSTOMER_ID` 是否等于 `0`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值
- 求值 `transactions_df[transactions_df.CUSTOMER_ID == 0]`：从 `transactions_df` 中按 `transactions_df.CUSTOMER_ID == 0` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `get_customer_spending_behaviour_features(transactions_df[transactions_df.CUSTOMER_ID == 0])`：调用 `get_customer_spending_behaviour_features`；它的返回值与副作用由此函数或类的定义决定，不能只凭名字推定数值。传入：位置参数1是 `transactions_df[transactions_df.CUSTOMER_ID == 0]`

- 调用 `get_customer_spending_behaviour_features` 时的位置参数1：`transactions_df[transactions_df.CUSTOMER_ID == 0]`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。

- `CUSTOMER_ID`：客户标识，不表示连续风险等级
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

**第 2 行**

```python
spending_behaviour_customer_0
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

变量 `spending_behaviour_customer_0` 当前引用的值。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [11]:
spending_behaviour_customer_0=get_customer_spending_behaviour_features(transactions_df[transactions_df.CUSTOMER_ID==0])
spending_behaviour_customer_0

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
TRANSACTION_ID,,,,,,,,,,,,,,,,,
1758,1758,2018-04-01 07:19:05,0,6076,123.59,26345,0,0,0,1,0,1.0,123.590000,1.0,123.590000,1.0,123.590000
8275,8275,2018-04-01 18:00:16,0,858,77.34,64816,0,0,0,1,0,2.0,100.465000,2.0,100.465000,2.0,100.465000
8640,8640,2018-04-01 19:02:02,0,6698,46.51,68522,0,0,0,1,0,3.0,82.480000,3.0,82.480000,3.0,82.480000
12169,12169,2018-04-02 08:51:06,0,6569,54.72,118266,1,0,0,0,0,3.0,59.523333,4.0,75.540000,4.0,75.540000
15764,15764,2018-04-02 14:05:38,0,7707,63.30,137138,1,0,0,0,0,4.0,60.467500,5.0,73.092000,5.0,73.092000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750390,1750390,2018-09-30 13:38:41,0,3096,38.23,15773921,182,0,0,1,0,5.0,64.388000,28.0,57.306429,89.0,63.097640
1750758,1750758,2018-09-30 14:10:21,0,9441,43.60,15775821,182,0,0,1,0,6.0,60.923333,29.0,56.833793,89.0,62.433933
1751039,1751039,2018-09-30 14:34:30,0,1138,69.69,15777270,182,0,0,1,0,7.0,62.175714,29.0,57.872414,90.0,62.514556


We can check that the new features are consistent with the customer profile (see the previous notebook). For customer 0, the mean amount was `mean_amount`=62.26, and the transaction frequency was `mean_nb_tx_per_day`=2.18. These values are indeed closely matched by the features `CUSTOMER_ID_NB_TX_30DAY_WINDOW` and `CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW`, especially after 30 days.

Let us now generate these features for all customers. This is straightforward using the Panda `groupby` and `apply` methods.

## 源码精读 · 原单元格 23

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 将同一消费特征函数扩展到所有客户。

**执行前：** transactions_df 是全体交易表，客户函数能处理一组。

**执行后：** 每位客户的加工结果组合成总表，再按时间排序和重置索引。

`.groupby('CUSTOMER_ID')` 按客户分组。`lambda x: ...` 创建临时函数，其中 `x` 是这一次传入的一位客户的小表，并不是单个金额。`.apply(...)` 每组调用一次，将结果组合。

人工追踪：原表四行依次 A1、B1、A2、B2；分组后函数先看到 `[A1,A2]`、再看到 `[B1,B2]`。每组独立算窗口，不串客户；拼回时可能形成包含分组键的层次索引，后续 `sort_values('TX_DATETIME')` 恢复按时间阅读的顺序，`reset_index(drop=True)` 丢弃旧索引并编号 0、1、2、3。业务列 `TRANSACTION_ID` 与重建的行号不是同一个概念。

新表赋回同名变量后，后续代码使用加工后的表；你没有必要为此在脑中保留每个随机金额，但必须知道每一列来自哪位客户、哪一个窗口。

**需要辨清的地方**

- 不同 pandas 版本对 groupby.apply 的分组列和索引处理可能不同；本解析保留原代码，不宣称已在当前环境跑通。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
%time transactions_df=transactions_df.groupby('CUSTOMER_ID').apply(lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1,7,30]))
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

IPython 的 %time 会执行后面的语句并记录耗时；计时结果取决于运行环境。 先计算右侧：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1, 7, 30])`。

把结果绑定到 `transactions_df`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `transactions_df.groupby('CUSTOMER_ID')`：按键划分组，先形成分组对象；后面的聚合或 apply 才决定输出。传入：位置参数1是 `'CUSTOMER_ID'`
- `lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1, 7, 30])`：定义临时函数，接收 `x`，调用时计算并返回 `get_customer_spending_behaviour_features(x, windows_size_in_days=[1, 7, 30])`
- 求值 `transactions_df.groupby('CUSTOMER_ID').apply(lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1, 7, 30]))`：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1, 7, 30])`

- 调用 `transactions_df.groupby('CUSTOMER_ID').apply` 时的位置参数1：`lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1, 7, 30])`；先求出该表达式再传入
- 调用 `transactions_df.groupby` 时的位置参数1：`'CUSTOMER_ID'`；先求出该表达式再传入
- 调用 `get_customer_spending_behaviour_features` 时的位置参数1：`x`；先求出该表达式再传入
- 调用 `get_customer_spending_behaviour_features` 时的具名参数 `windows_size_in_days` 接收 `[1, 7, 30]`

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易
- `windows_size_in_days`：需要分别计算的多个窗口天数

**第 2 行**

```python
transactions_df=transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：重置行索引；drop=True 丢弃旧索引，否则旧索引通常转为列。传入：`drop=True` 是具名参数。

把结果绑定到 `transactions_df`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `transactions_df.sort_values('TX_DATETIME')`：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_DATETIME'`
- 求值 `transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)`：重置行索引；drop=True 丢弃旧索引，否则旧索引通常转为列。传入：`drop=True` 是具名参数

- 调用 `transactions_df.sort_values('TX_DATETIME').reset_index` 时的具名参数 `drop` 接收 `True`
- 调用 `transactions_df.sort_values` 时的位置参数1：`'TX_DATETIME'`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。

- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [12]:
%time transactions_df=transactions_df.groupby('CUSTOMER_ID').apply(lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1,7,30]))
transactions_df=transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)


CPU times: user 1min 2s, sys: 1.21 s, total: 1min 3s
Wall time: 1min 7s


## 源码精读 · 原单元格 24

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 查看加入客户消费特征后的总表。

**执行前：** 上一格已将加工结果赋回 transactions_df。

**执行后：** 只是显示当前对象；没有再计算一轮窗口。

检查三对字段的列名：每个天数对应 NB_TX 与 AVG_AMOUNT。表很宽时 Notebook 可能用省略号隐藏中间列，省略号不等于字段不存在。底部形状表示全表范围，屏幕上展示的前后几行只是格式化摘要。要理解一个具体结果，应回到同一客户的时间序列逐笔手算，而不是从不同客户的相邻显示行推断窗口。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
transactions_df
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

变量 `transactions_df` 当前引用的值。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [13]:
transactions_df

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0,1,1,1.0,57.160000,1.0,57.160000,1.0,57.160000
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0,1,1,1.0,81.510000,1.0,81.510000,1.0,81.510000
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0,1,1,1.0,146.000000,1.0,146.000000,1.0,146.000000
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0,1,1,1.0,64.490000,1.0,64.490000,1.0,64.490000
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0,1,1,1.0,50.990000,1.0,50.990000,1.0,50.990000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754150,1754150,2018-09-30 23:56:36,161,655,54.24,15810996,182,0,0,1,0,2.0,75.280000,12.0,67.047500,72.0,69.521111
1754151,1754151,2018-09-30 23:57:38,4342,6181,1.23,15811058,182,0,0,1,0,1.0,1.230000,21.0,22.173810,93.0,24.780753
1754152,1754152,2018-09-30 23:58:21,618,1502,6.62,15811101,182,0,0,1,0,5.0,7.368000,21.0,7.400476,65.0,7.864462
1754153,1754153,2018-09-30 23:59:52,4056,3067,55.40,15811192,182,0,0,1,0,3.0,100.696667,16.0,107.052500,51.0,102.919608


## Terminal ID transformations

Finally, let us proceed with the terminal ID transformations. The main goal will be to extract a *risk score*, that assesses the exposure of a given terminal ID to fraudulent transactions. The risk score will be defined as the average number of fraudulent transactions that occurred on a terminal ID over a time window. As for customer ID transformations, we will use three window sizes, of 1, 7, and 30 days.

Contrary to customer ID transformations, the time windows will not directly precede a given transaction. Instead, they will be shifted back by a *delay period*. The delay period accounts for the fact that, in practice, the fraudulent transactions are only discovered after a fraud investigation or a customer complaint. Hence, the fraudulent labels, which are needed to compute the risk score, are only available after this delay period. To a first approximation, this delay period will be set to one week. The motivations for the delay period will be further argued in [Chapter 5, Validation strategies](Validation_Strategies). 

Let us perform the computation of the risk scores by defining a `get_count_risk_rolling_window` function. The function takes as inputs the DataFrame of transactions for a given terminal ID, the delay period, and a list of window sizes. In the first stage, the number of transactions and fraudulent transactions are computed for the delay period (`NB_TX_DELAY` and `NB_FRAUD_DELAY`). In the second stage, the number of transactions and fraudulent transactions are computed for each window size plus the delay period (`NB_TX_DELAY_WINDOW` and `NB_FRAUD_DELAY_WINDOW`). The number of transactions and fraudulent transactions that occurred for a given window size, shifted back by the delay period, is then obtained by simply computing the differences of the quantities obtained for the delay period, and the window size plus delay period:

```
NB_FRAUD_WINDOW=NB_FRAUD_DELAY_WINDOW-NB_FRAUD_DELAY
NB_TX_WINDOW=NB_TX_DELAY_WINDOW-NB_TX_DELAY
```

The risk score is finally obtained by computing the proportion of fraudulent transactions for each window size (or 0 if no transaction occurred for the given window):

```
RISK_WINDOW=NB_FRAUD_WINDOW/NB_TX_WINDOW
```

Additionally to the risk score, the function also returns the number of transactions for each window size. This results in the addition of six new features: The risk and number of transactions, for three window sizes.


## 源码精读 · 原单元格 27

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 定义使用延迟标签的终端交易量与欺诈比例特征。

**执行前：** 输入一台终端的交易表，标签在模拟数据中已知，但业务假设需等待 delay_period 天才能获得。

**执行后：** 实际调用返回原行加 1、7、30 天窗口的交易数量和风险比例，共六列。

核心不是直接对过去一天 `TX_FRAUD.mean()`。如果欺诈反馈晚七天，当下不能使用最近七天的真实标签。代码先分别计算 `(t-7天,t]` 的短窗口，以及 `(t-(7+w)天,t]` 的长窗口，再用长减短，留下 `(t-7-w天,t-7天]` 的可见区域。

用人为设置的窗口计数演示 `w=1`：长八天内 20 笔、其中 3 笔欺诈；短七天内 16 笔、其中 2 笔欺诈。相减得可见一天 4 笔、1 笔欺诈，风险值 `1/4=0.25`。不是 `3/20`，也不是 `(3/20)-(2/16)`；要先减计数，再求比例。

`NB_FRAUD_DELAY` 和 `NB_TX_DELAY` 在三个窗口间共用，故放在循环外。循环体构造不同长窗口，得到 `NB_FRAUD_WINDOW`、`NB_TX_WINDOW`、`RISK_WINDOW`。字符串拼接 `feature+'_RISK_'+str(window_size)+'DAY_WINDOW'` 将参数值装进列名，`feature` 默认是 `'TERMINAL_ID'`；列名中的文字不代表又执行了一次数据库查询。

时间索引用于滚动；最后把索引恢复成交易 ID，再执行 `fillna(0,inplace=True)`。若可见区间没有交易，风险可能出现 `0/0` 的缺失值，被填成 0。这个 0 表示此实现采用的默认数值，不证明终端客观上完全安全。而且填充作用于传入加工表的所有缺失项，不只新生成的风险列。

这格只是定义。下一格真实调用时，返回表才会带新列。窗口边界、标签延迟和分组键决定了它是否符合时间因果关系，不能只因为列名写了 RISK 就当作没有泄漏。

**需要辨清的地方**

- 同一终端内排序和正确的反馈可见性是假设前提。
- fillna(0) 会把未知或无历史与观测到零欺诈混在同一数值里。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
def get_count_risk_rolling_window(terminal_transactions, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"):
```

执行位置：此处是函数 `get_count_risk_rolling_window` 的定义签名；默认值表达式在到达定义时求值，函数体等待以后调用

定义函数 `get_count_risk_rolling_window(terminal_transactions, delay_period=7, windows_size_in_days=[1, 7, 30], feature='TERMINAL_ID')`。冒号后缩进的代码是它的函数体。

到达此定义时求值默认参数等定义期表达式，建立函数对象并绑定名称；不会因此执行函数体。函数体说明仅在以后调用并走到相应路径时成立。

- 求值 `[1, 7, 30]`：含 3 项的列表：常量 `1`；常量 `7`；常量 `30`

- 定义参数 `terminal_transactions`，本签名未给默认值，调用者需按参数规则提供
- 定义参数 `delay_period` 的默认表达式 `7`；它在定义时求值一次，调用未提供该参数时才采用这个默认对象
- 定义参数 `windows_size_in_days` 的默认表达式 `[1, 7, 30]`；它在定义时求值一次，调用未提供该参数时才采用这个默认对象
- 定义参数 `feature` 的默认表达式 `'TERMINAL_ID'`；它在定义时求值一次，调用未提供该参数时才采用这个默认对象

- 函数定义签名中的 `参数名=表达式` 指定默认值；默认表达式在定义时求值，不能当成一次实际传参或函数体赋值。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- def 后的外层圆括号列出定义参数，不是调用；其中默认表达式若含另一个函数调用，它仍在定义时求值。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

- `delay_period`：反馈延迟天数
- `windows_size_in_days`：需要分别计算的多个窗口天数

**第 3 行**

```python
    terminal_transactions=terminal_transactions.sort_values('TX_DATETIME')
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_DATETIME'`。

把结果绑定到 `terminal_transactions`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `terminal_transactions.sort_values('TX_DATETIME')`：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_DATETIME'`

- 调用 `terminal_transactions.sort_values` 时的位置参数1：`'TX_DATETIME'`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 5 行**

```python
    terminal_transactions.index=terminal_transactions.TX_DATETIME
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：访问 `terminal_transactions` 的 `TX_DATETIME` 属性或成员。

把结果保存到 `terminal_transactions.index` 指定的对象属性；这会改变该对象的状态。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `TX_DATETIME`：交易发生的日期时间

**第 7 行**

```python
    NB_FRAUD_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').sum()
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：对指定数据或窗口求和；0/1标签求和可统计欺诈笔数。这里没有显式传入参数。

把结果绑定到 `NB_FRAUD_DELAY`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `terminal_transactions['TX_FRAUD']`：从 `terminal_transactions` 中按 `'TX_FRAUD'` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `str(delay_period)`：按 `str` 对输入进行构造或类型转换。传入：位置参数1是 `delay_period`
- 求值 `str(delay_period) + 'd'`：先计算 `str(delay_period)` 与 `'d'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period) + 'd')`：建立滚动窗口视图，窗口按行数还是时间取决于参数；后面的 sum/count 等才计算值。传入：位置参数1是 `str(delay_period) + 'd'`
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period) + 'd').sum()`：对指定数据或窗口求和；0/1标签求和可统计欺诈笔数。这里没有显式传入参数

- 调用 `terminal_transactions['TX_FRAUD'].rolling` 时的位置参数1：`str(delay_period) + 'd'`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`delay_period`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `delay_period`：反馈延迟天数

**第 8 行**

```python
    NB_TX_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').count()
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：统计非缺失记录数量；不要自动等同包含缺失的所有行数。这里没有显式传入参数。

把结果绑定到 `NB_TX_DELAY`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `terminal_transactions['TX_FRAUD']`：从 `terminal_transactions` 中按 `'TX_FRAUD'` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `str(delay_period)`：按 `str` 对输入进行构造或类型转换。传入：位置参数1是 `delay_period`
- 求值 `str(delay_period) + 'd'`：先计算 `str(delay_period)` 与 `'d'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period) + 'd')`：建立滚动窗口视图，窗口按行数还是时间取决于参数；后面的 sum/count 等才计算值。传入：位置参数1是 `str(delay_period) + 'd'`
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period) + 'd').count()`：统计非缺失记录数量；不要自动等同包含缺失的所有行数。这里没有显式传入参数

- 调用 `terminal_transactions['TX_FRAUD'].rolling` 时的位置参数1：`str(delay_period) + 'd'`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`delay_period`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `delay_period`：反馈延迟天数

**第 10 行**

```python
    for window_size in windows_size_in_days:
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

遍历 `windows_size_in_days`，每次把取出的项交给 `window_size`。

对每项执行缩进体；循环变量通常被下一项覆盖，累计结果需另放列表、表格或计数器。空可迭代对象不会执行循环体。

- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度
- `windows_size_in_days`：需要分别计算的多个窗口天数

**第 12 行**

```python
        NB_FRAUD_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').sum()
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：对指定数据或窗口求和；0/1标签求和可统计欺诈笔数。这里没有显式传入参数。

把结果绑定到 `NB_FRAUD_DELAY_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `terminal_transactions['TX_FRAUD']`：从 `terminal_transactions` 中按 `'TX_FRAUD'` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `delay_period + window_size`：先计算 `delay_period` 与 `window_size`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `str(delay_period + window_size)`：按 `str` 对输入进行构造或类型转换。传入：位置参数1是 `delay_period + window_size`
- 求值 `str(delay_period + window_size) + 'd'`：先计算 `str(delay_period + window_size)` 与 `'d'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period + window_size) + 'd')`：建立滚动窗口视图，窗口按行数还是时间取决于参数；后面的 sum/count 等才计算值。传入：位置参数1是 `str(delay_period + window_size) + 'd'`
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period + window_size) + 'd').sum()`：对指定数据或窗口求和；0/1标签求和可统计欺诈笔数。这里没有显式传入参数

- 调用 `terminal_transactions['TX_FRAUD'].rolling` 时的位置参数1：`str(delay_period + window_size) + 'd'`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`delay_period + window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `delay_period`：反馈延迟天数
- `window_size`：本轮滚动窗口长度

**第 13 行**

```python
        NB_TX_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').count()
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：统计非缺失记录数量；不要自动等同包含缺失的所有行数。这里没有显式传入参数。

把结果绑定到 `NB_TX_DELAY_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `terminal_transactions['TX_FRAUD']`：从 `terminal_transactions` 中按 `'TX_FRAUD'` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `delay_period + window_size`：先计算 `delay_period` 与 `window_size`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `str(delay_period + window_size)`：按 `str` 对输入进行构造或类型转换。传入：位置参数1是 `delay_period + window_size`
- 求值 `str(delay_period + window_size) + 'd'`：先计算 `str(delay_period + window_size)` 与 `'d'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period + window_size) + 'd')`：建立滚动窗口视图，窗口按行数还是时间取决于参数；后面的 sum/count 等才计算值。传入：位置参数1是 `str(delay_period + window_size) + 'd'`
- 求值 `terminal_transactions['TX_FRAUD'].rolling(str(delay_period + window_size) + 'd').count()`：统计非缺失记录数量；不要自动等同包含缺失的所有行数。这里没有显式传入参数

- 调用 `terminal_transactions['TX_FRAUD'].rolling` 时的位置参数1：`str(delay_period + window_size) + 'd'`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`delay_period + window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `delay_period`：反馈延迟天数
- `window_size`：本轮滚动窗口长度

**第 15 行**

```python
        NB_FRAUD_WINDOW=NB_FRAUD_DELAY_WINDOW-NB_FRAUD_DELAY
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：先取得 变量 `NB_FRAUD_DELAY_WINDOW` 当前引用的值，再与 变量 `NB_FRAUD_DELAY` 当前引用的值 做“减”；数组或Series时通常逐元素计算，形状或索引须兼容。

把结果绑定到 `NB_FRAUD_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `NB_FRAUD_DELAY_WINDOW - NB_FRAUD_DELAY`：先取得 变量 `NB_FRAUD_DELAY_WINDOW` 当前引用的值，再与 变量 `NB_FRAUD_DELAY` 当前引用的值 做“减”；数组或Series时通常逐元素计算，形状或索引须兼容

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 16 行**

```python
        NB_TX_WINDOW=NB_TX_DELAY_WINDOW-NB_TX_DELAY
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：先取得 变量 `NB_TX_DELAY_WINDOW` 当前引用的值，再与 变量 `NB_TX_DELAY` 当前引用的值 做“减”；数组或Series时通常逐元素计算，形状或索引须兼容。

把结果绑定到 `NB_TX_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `NB_TX_DELAY_WINDOW - NB_TX_DELAY`：先取得 变量 `NB_TX_DELAY_WINDOW` 当前引用的值，再与 变量 `NB_TX_DELAY` 当前引用的值 做“减”；数组或Series时通常逐元素计算，形状或索引须兼容

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 18 行**

```python
        RISK_WINDOW=NB_FRAUD_WINDOW/NB_TX_WINDOW
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：先取得 变量 `NB_FRAUD_WINDOW` 当前引用的值，再与 变量 `NB_TX_WINDOW` 当前引用的值 做“除以”；数组或Series时通常逐元素计算，形状或索引须兼容。

把结果绑定到 `RISK_WINDOW`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `NB_FRAUD_WINDOW / NB_TX_WINDOW`：先取得 变量 `NB_FRAUD_WINDOW` 当前引用的值，再与 变量 `NB_TX_WINDOW` 当前引用的值 做“除以”；数组或Series时通常逐元素计算，形状或索引须兼容

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 20 行**

```python
        terminal_transactions[feature+'_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `NB_TX_WINDOW`。

再把结果写到 `terminal_transactions[feature + '_NB_TX_' + str(window_size) + 'DAY_WINDOW']` 指定的位置或列；若是DataFrame列赋值，需要检查行数、索引对齐和是否覆盖已有列。

- 求值 `list(NB_TX_WINDOW)`：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `NB_TX_WINDOW`

- 调用 `list` 时的位置参数1：`NB_TX_WINDOW`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度

**第 21 行**

```python
        terminal_transactions[feature+'_RISK_'+str(window_size)+'DAY_WINDOW']=list(RISK_WINDOW)
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体；属于循环体，仅在本轮从 `windows_size_in_days` 取得一项赋给 `window_size`后走到本行时执行

先计算右侧：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `RISK_WINDOW`。

再把结果写到 `terminal_transactions[feature + '_RISK_' + str(window_size) + 'DAY_WINDOW']` 指定的位置或列；若是DataFrame列赋值，需要检查行数、索引对齐和是否覆盖已有列。

- 求值 `list(RISK_WINDOW)`：按 `list` 对输入进行构造或类型转换。传入：位置参数1是 `RISK_WINDOW`

- 调用 `list` 时的位置参数1：`RISK_WINDOW`；先求出该表达式再传入
- 调用 `str` 时的位置参数1：`window_size`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `window_size`：本轮滚动窗口长度

**第 23 行**

```python
    terminal_transactions.index=terminal_transactions.TRANSACTION_ID
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

先计算右侧：访问 `terminal_transactions` 的 `TRANSACTION_ID` 属性或成员。

把结果保存到 `terminal_transactions.index` 指定的对象属性；这会改变该对象的状态。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `TRANSACTION_ID`：交易标识

**第 25 行**

```python
    # Replace NA values with 0 (all undefined risk scores where NB_TX_WINDOW is 0) 
```

执行位置：本行是注释，不执行。所在上下文：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

原作者的说明性注释：Replace NA values with 0 (all undefined risk scores where NB_TX_WINDOW is 0)

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 26 行**

```python
    terminal_transactions.fillna(0,inplace=True)
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

按参数填补缺失值；inplace=True 会改变接收对象，返回值不可当成新表。传入：位置参数1是 `0`；`inplace=True` 是具名参数。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `terminal_transactions.fillna(0, inplace=True)`：按参数填补缺失值；inplace=True 会改变接收对象，返回值不可当成新表。传入：位置参数1是 `0`；`inplace=True` 是具名参数

- 调用 `terminal_transactions.fillna` 时的位置参数1：`0`；先求出该表达式再传入
- 调用 `terminal_transactions.fillna` 时的具名参数 `inplace` 接收 `True`

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 28 行**

```python
    return terminal_transactions
```

执行位置：仅在调用 `get_count_risk_rolling_window` 并执行到本行时发生；仅执行 def 不执行这段函数体

计算要返回的内容：变量 `terminal_transactions` 当前引用的值。

立即结束本次函数调用，把这个结果交回调用处；这次函数调用中后面的语句不再执行。返回表不等于自动写入磁盘。

- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [14]:
def get_count_risk_rolling_window(terminal_transactions, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"):
    
    terminal_transactions=terminal_transactions.sort_values('TX_DATETIME')
    
    terminal_transactions.index=terminal_transactions.TX_DATETIME
    
    NB_FRAUD_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').sum()
    NB_TX_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').count()
    
    for window_size in windows_size_in_days:
    
        NB_FRAUD_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').sum()
        NB_TX_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').count()
    
        NB_FRAUD_WINDOW=NB_FRAUD_DELAY_WINDOW-NB_FRAUD_DELAY
        NB_TX_WINDOW=NB_TX_DELAY_WINDOW-NB_TX_DELAY
    
        RISK_WINDOW=NB_FRAUD_WINDOW/NB_TX_WINDOW
        
        terminal_transactions[feature+'_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
        terminal_transactions[feature+'_RISK_'+str(window_size)+'DAY_WINDOW']=list(RISK_WINDOW)
        
    terminal_transactions.index=terminal_transactions.TRANSACTION_ID
    
    # Replace NA values with 0 (all undefined risk scores where NB_TX_WINDOW is 0) 
    terminal_transactions.fillna(0,inplace=True)
    
    return terminal_transactions


## 源码精读 · 原单元格 28

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 显示真实标签为 1 的交易。

**执行前：** 全表包含 TX_FRAUD。

**执行后：** 产生欺诈行子表供阅读，未改变 transactions_df 的行集合。

先得到 `TX_FRAUD==1` 的布尔序列，再筛选整表。这个动作是在已经有标签的教学数据里查看案例，不是模型作出了“欺诈”预测。看原书历史输出时，应将标签列和以后生成的预测分数分开：标签是事后真值，分数是模型根据输入特征给出的判断。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
transactions_df[transactions_df.TX_FRAUD==1]
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

从 `transactions_df` 中按 `transactions_df.TX_FRAUD == 1` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `transactions_df.TX_FRAUD == 1`：判断`transactions_df.TX_FRAUD` 是否等于 `1`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值
- 求值 `transactions_df[transactions_df.TX_FRAUD == 1]`：从 `transactions_df` 中按 `transactions_df.TX_FRAUD == 1` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则

- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。

- `TX_FRAUD`：0/1 欺诈标签，属于监督目标或事后评估信息
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [15]:
transactions_df[transactions_df.TX_FRAUD==1]

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
3527,3527,2018-04-01 10:17:43,3774,3059,225.41,37063,0,1,1,1,0,3.0,158.073333,3.0,158.073333,3.0,158.073333
5789,5790,2018-04-01 13:31:48,4944,6050,222.26,48708,0,1,1,1,0,2.0,127.605000,2.0,127.605000,2.0,127.605000
6549,6549,2018-04-01 14:42:02,4625,9102,226.40,52922,0,1,1,1,0,4.0,167.165000,4.0,167.165000,4.0,167.165000
9583,9583,2018-04-02 01:01:05,3814,6893,59.15,90065,1,1,3,0,1,6.0,29.138333,6.0,29.138333,6.0,29.138333
10356,10355,2018-04-02 05:03:35,2513,1143,222.04,104615,1,1,1,0,1,5.0,123.740000,5.0,123.740000,5.0,123.740000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1753524,1753524,2018-09-30 19:51:48,1671,3192,128.60,15796308,182,1,3,1,0,6.0,138.358333,25.0,106.957200,82.0,75.621341
1753600,1753600,2018-09-30 20:09:00,4166,632,17.39,15797340,182,1,2,1,0,3.0,19.766667,19.0,15.984737,86.0,15.846512
1753673,1753673,2018-09-30 20:30:52,4097,1558,24.04,15798652,182,1,2,1,0,3.0,23.050000,16.0,40.440625,63.0,41.877460
1754014,1754014,2018-09-30 22:27:04,100,8604,73.85,15805624,182,1,3,1,0,2.0,48.010000,26.0,30.384231,103.0,23.627184


Let us compute these six features for the first terminal ID containing at least one fraud:

## 源码精读 · 原单元格 30

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 原作者尝试查看某个终端 ID，但注释和筛选条件不一致。

**执行前：** 有交易表和真实标签。

**执行后：** 若索引访问成功，表达式返回一个终端编号；它不保证该编号来自欺诈交易。

按代码读：`TX_FRAUD==0` 筛出正常交易，然后取它们的 `TERMINAL_ID` 列，最后 `[0]` 再索引。注释说 first terminal ID that contains frauds，实际条件却是 0，因此不能将注释照译成执行结果。

再留意 Series 的 `[0]`：它在相应索引规则下可能按标签 0 取值，而非无条件取筛选后的第一项。若筛选结果没有标签 0，可能失败。若教学目的是明确取第一项，可以另外讨论 `.iloc[0]`；本页保留原始代码，不替作者偷偷修正，也不把历史输出中的编号宣称为本次验证结果。

**需要辨清的地方**

- TX_FRAUD==0 与原注释不一致；[0] 的索引含义也需区分。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
# Get the first terminal ID that contains frauds
```

执行位置：本行是注释，不执行。所在上下文：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

原作者的说明性注释：Get the first terminal ID that contains frauds

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。

**第 2 行**

```python
transactions_df[transactions_df.TX_FRAUD==0].TERMINAL_ID[0]
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

从 `transactions_df[transactions_df.TX_FRAUD == 0].TERMINAL_ID` 中按 `0` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `transactions_df.TX_FRAUD == 0`：判断`transactions_df.TX_FRAUD` 是否等于 `0`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值
- 求值 `transactions_df[transactions_df.TX_FRAUD == 0]`：从 `transactions_df` 中按 `transactions_df.TX_FRAUD == 0` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `transactions_df[transactions_df.TX_FRAUD == 0].TERMINAL_ID[0]`：从 `transactions_df[transactions_df.TX_FRAUD == 0].TERMINAL_ID` 中按 `0` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则

- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。

- `TERMINAL_ID`：终端标识
- `TX_FRAUD`：0/1 欺诈标签，属于监督目标或事后评估信息
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [16]:
# Get the first terminal ID that contains frauds
transactions_df[transactions_df.TX_FRAUD==0].TERMINAL_ID[0]

3156

## 源码精读 · 原单元格 31

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 在终端 3059 的交易上试算风险窗口。

**执行前：** 终端函数已定义，原表应含该终端的记录。

**执行后：** 返回这个终端的交易子表及六个新列，供单个案例检查。

`TERMINAL_ID==3059` 只是教材选定的一个例子，不是筛选最高风险终端。`delay_period=7` 明确给标签等待期；`windows_size_in_days=[1,7,30]` 指等待期之前分别看多长的历史。

审阅一行时同时读时间、笔数窗口和风险窗口。例如人工示意 NB_TX=4、RISK=0.25 可解释为该可见区间内四笔中一笔标为欺诈；如果风险为 0 且笔数为 0，则是没有历史和填充值的情况，含义不同。应先在单终端试算里理解差别，再读下一格的全体分组。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
get_count_risk_rolling_window(transactions_df[transactions_df.TERMINAL_ID==3059], delay_period=7, windows_size_in_days=[1,7,30])
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

调用 `get_count_risk_rolling_window`；它的返回值与副作用由此函数或类的定义决定，不能只凭名字推定数值。传入：位置参数1是 `transactions_df[transactions_df.TERMINAL_ID == 3059]`；`delay_period=7` 是具名参数；`windows_size_in_days=[1, 7, 30]` 是具名参数。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `transactions_df.TERMINAL_ID == 3059`：判断`transactions_df.TERMINAL_ID` 是否等于 `3059`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值
- 求值 `transactions_df[transactions_df.TERMINAL_ID == 3059]`：从 `transactions_df` 中按 `transactions_df.TERMINAL_ID == 3059` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `[1, 7, 30]`：含 3 项的列表：常量 `1`；常量 `7`；常量 `30`
- 求值 `get_count_risk_rolling_window(transactions_df[transactions_df.TERMINAL_ID == 3059], delay_period=7, windows_size_in_days=[1, 7, 30])`：调用 `get_count_risk_rolling_window`；它的返回值与副作用由此函数或类的定义决定，不能只凭名字推定数值。传入：位置参数1是 `transactions_df[transactions_df.TERMINAL_ID == 3059]`；`delay_period=7` 是具名参数；`windows_size_in_days=[1, 7, 30]` 是具名参数

- 调用 `get_count_risk_rolling_window` 时的位置参数1：`transactions_df[transactions_df.TERMINAL_ID == 3059]`；先求出该表达式再传入
- 调用 `get_count_risk_rolling_window` 时的具名参数 `delay_period` 接收 `7`
- 调用 `get_count_risk_rolling_window` 时的具名参数 `windows_size_in_days` 接收 `[1, 7, 30]`

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

- `TERMINAL_ID`：终端标识
- `delay_period`：反馈延迟天数
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易
- `windows_size_in_days`：需要分别计算的多个窗口天数

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [17]:
get_count_risk_rolling_window(transactions_df[transactions_df.TERMINAL_ID==3059], delay_period=7, windows_size_in_days=[1,7,30])

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,...,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW,TERMINAL_ID_NB_TX_1DAY_WINDOW,TERMINAL_ID_RISK_1DAY_WINDOW,TERMINAL_ID_NB_TX_7DAY_WINDOW,TERMINAL_ID_RISK_7DAY_WINDOW,TERMINAL_ID_NB_TX_30DAY_WINDOW,TERMINAL_ID_RISK_30DAY_WINDOW
TRANSACTION_ID,,,,,,,,,,,,,,,,,,,,,
3527,3527,2018-04-01 10:17:43,3774,3059,225.41,37063,0,1,1,1,...,3.0,158.073333,3.0,158.073333,0.0,0.0,0.0,0.0,0.0,0.0
4732,4732,2018-04-01 11:59:14,55,3059,36.28,43154,0,0,0,1,...,2.0,35.670000,2.0,35.670000,0.0,0.0,0.0,0.0,0.0,0.0
16216,16216,2018-04-02 14:47:34,4879,3059,105.00,139654,1,0,0,0,...,10.0,76.010000,10.0,76.010000,0.0,0.0,0.0,0.0,0.0,0.0
18249,18249,2018-04-02 19:08:10,2263,3059,90.89,155290,1,0,0,0,...,7.0,50.458571,7.0,50.458571,0.0,0.0,0.0,0.0,0.0,0.0
26512,26512,2018-04-03 15:44:49,4879,3059,58.51,229489,2,0,0,0,...,14.0,71.070000,14.0,71.070000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1697944,1697944,2018-09-25 05:32:56,402,3059,57.30,15312776,177,0,0,0,...,14.0,65.167857,46.0,68.163261,1.0,0.0,9.0,0.0,36.0,0.0
1701971,1701971,2018-09-25 12:30:54,1035,3059,7.56,15337854,177,0,0,0,...,23.0,7.052174,107.0,6.763738,2.0,0.0,10.0,0.0,36.0,0.0
1704512,1704512,2018-09-25 16:37:41,1519,3059,35.79,15352661,177,0,0,0,...,7.0,41.404286,30.0,46.780000,1.0,0.0,9.0,0.0,36.0,0.0


We can check that the first fraud occurred on the 2018/09/10, and that risk scores only start being counted with a one-week delay. 

Let us finally generate these features for all terminals. This is straightforward using the Panda `groupby` and `apply` methods. 

## 源码精读 · 原单元格 33

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 为所有终端的交易添加延迟风险特征。

**执行前：** 全体交易已有客户特征；终端函数可接收一个终端子表。

**执行后：** 每行新增相应终端的六列历史特征，总表重新按时间排好并重置行索引。

与客户分组那格对照读：分组键改成 `'TERMINAL_ID'`，函数改成 `get_count_risk_rolling_window`，每个 `x` 现在是一台终端接收的多位客户交易。客户特征不会因此变成终端特征；已有列跟着原行保留，新增列才描述终端。

人工四行 A在T1、B在T2、C在T1、A在T2，分组得到 T1 的第一/三行和 T2 的第二/四行。只在各组内计算可见历史。最终按时间排序，再 `reset_index(drop=True)`。这一步的关键学习能力是“改变观察单位”：同一笔交易可以同时具有客户历史与终端历史。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
%time transactions_df=transactions_df.groupby('TERMINAL_ID').apply(lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"))
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

IPython 的 %time 会执行后面的语句并记录耗时；计时结果取决于运行环境。 先计算右侧：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1, 7, 30], feature='TERMINAL_ID')`。

把结果绑定到 `transactions_df`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `transactions_df.groupby('TERMINAL_ID')`：按键划分组，先形成分组对象；后面的聚合或 apply 才决定输出。传入：位置参数1是 `'TERMINAL_ID'`
- `lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1, 7, 30], feature='TERMINAL_ID')`：定义临时函数，接收 `x`，调用时计算并返回 `get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1, 7, 30], feature='TERMINAL_ID')`
- 求值 `transactions_df.groupby('TERMINAL_ID').apply(lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1, 7, 30], feature='TERMINAL_ID'))`：将给定函数应用到指定对象的元素、行列或分组；准确粒度由接收对象和 axis 决定。传入：位置参数1是 `lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1, 7, 30], feature='TERMINAL_ID')`

- 调用 `transactions_df.groupby('TERMINAL_ID').apply` 时的位置参数1：`lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1, 7, 30], feature='TERMINAL_ID')`；先求出该表达式再传入
- 调用 `transactions_df.groupby` 时的位置参数1：`'TERMINAL_ID'`；先求出该表达式再传入
- 调用 `get_count_risk_rolling_window` 时的位置参数1：`x`；先求出该表达式再传入
- 调用 `get_count_risk_rolling_window` 时的具名参数 `delay_period` 接收 `7`
- 调用 `get_count_risk_rolling_window` 时的具名参数 `windows_size_in_days` 接收 `[1, 7, 30]`
- 调用 `get_count_risk_rolling_window` 时的具名参数 `feature` 接收 `'TERMINAL_ID'`

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

- `delay_period`：反馈延迟天数
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易
- `windows_size_in_days`：需要分别计算的多个窗口天数

**第 2 行**

```python
transactions_df=transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：重置行索引；drop=True 丢弃旧索引，否则旧索引通常转为列。传入：`drop=True` 是具名参数。

把结果绑定到 `transactions_df`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `transactions_df.sort_values('TX_DATETIME')`：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_DATETIME'`
- 求值 `transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)`：重置行索引；drop=True 丢弃旧索引，否则旧索引通常转为列。传入：`drop=True` 是具名参数

- 调用 `transactions_df.sort_values('TX_DATETIME').reset_index` 时的具名参数 `drop` 接收 `True`
- 调用 `transactions_df.sort_values` 时的位置参数1：`'TX_DATETIME'`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。

- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [18]:
%time transactions_df=transactions_df.groupby('TERMINAL_ID').apply(lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"))
transactions_df=transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)


CPU times: user 2min 27s, sys: 2.23 s, total: 2min 29s
Wall time: 2min 41s


## 源码精读 · 原单元格 34

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 查看最终特征表。

**执行前：** 客户、终端与时间特征均已添加。

**执行后：** 仅显示；不保存、不训练。

此时可以把一行读成一笔交易的“当前信息＋客户行为背景＋终端历史背景”。标签列仍用于后续监督训练，不能混进输入列。表里有某个字段，不等于训练时一定会使用它；真正的模型输入由 BaselineModeling 的 input_features 明确挑选。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
transactions_df
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

变量 `transactions_df` 当前引用的值。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [19]:
transactions_df

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,...,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW,TERMINAL_ID_NB_TX_1DAY_WINDOW,TERMINAL_ID_RISK_1DAY_WINDOW,TERMINAL_ID_NB_TX_7DAY_WINDOW,TERMINAL_ID_RISK_7DAY_WINDOW,TERMINAL_ID_NB_TX_30DAY_WINDOW,TERMINAL_ID_RISK_30DAY_WINDOW
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0,1,...,1.0,57.160000,1.0,57.160000,0.0,0.0,0.0,0.0,0.0,0.00000
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0,1,...,1.0,81.510000,1.0,81.510000,0.0,0.0,0.0,0.0,0.0,0.00000
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0,1,...,1.0,146.000000,1.0,146.000000,0.0,0.0,0.0,0.0,0.0,0.00000
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0,1,...,1.0,64.490000,1.0,64.490000,0.0,0.0,0.0,0.0,0.0,0.00000
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0,1,...,1.0,50.990000,1.0,50.990000,0.0,0.0,0.0,0.0,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754150,1754150,2018-09-30 23:56:36,161,655,54.24,15810996,182,0,0,1,...,12.0,67.047500,72.0,69.521111,1.0,0.0,4.0,0.0,28.0,0.00000
1754151,1754151,2018-09-30 23:57:38,4342,6181,1.23,15811058,182,0,0,1,...,21.0,22.173810,93.0,24.780753,1.0,0.0,9.0,0.0,39.0,0.00000
1754152,1754152,2018-09-30 23:58:21,618,1502,6.62,15811101,182,0,0,1,...,21.0,7.400476,65.0,7.864462,1.0,0.0,5.0,0.0,33.0,0.00000
1754153,1754153,2018-09-30 23:59:52,4056,3067,55.40,15811192,182,0,0,1,...,16.0,107.052500,51.0,102.919608,1.0,0.0,6.0,0.0,28.0,0.00000


## Saving of dataset

Let us finally save the dataset, split into daily batches, using the pickle format. 

## 源码精读 · 原单元格 36

**逐格专项补注。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 按天把加工后的交易表保存为 pickle 文件。

**执行前：** transactions_df 已有 TX_TIME_DAYS、TX_TIME_SECONDS 和新增特征。

**执行后：** 实际运行会创建输出目录并为每个日编号写文件；这是写盘动作，本精读未执行。

`range(max_day+1)` 包含 0 到最大天编号；加 1 是因为 Python 范围的右端不包含。每轮用 `TX_TIME_DAYS==day` 取当天交易，再按秒数排序。`start_date+timedelta(days=day)` 把日编号换成日期，`.strftime('%Y-%m-%d')` 生成文件名的一部分。

人工演示最大日编号为 2：循环依次处理 0、1、2，日期从起点依次加 0、1、2 天。`to_pickle(...,protocol=4)` 将当天的表序列化，和 `return` 把对象交回调用者完全不同。若文件已存在，实际写入可能覆盖；此页面只是阅读，不会触发保存。

留意原笔记本这里输出到 `./simulated-data-transformed/`，下一份基准建模读取的是仓库数据路径 `./simulated-data-transformed/data/`。两者用途和目录层级要核对，不能以为只运行完本格，另一页默认路径一定立刻对应你的输出。

### 按物理代码行精读（不含纯空行）

**第 1 行**

```python
DIR_OUTPUT = "./simulated-data-transformed/"
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：字符串 `./simulated-data-transformed/`（文字，不是同名变量）。

把结果绑定到 `DIR_OUTPUT`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。

**第 3 行**

```python
if not os.path.exists(DIR_OUTPUT):
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

判断条件：对条件取逻辑否定，作用对象是 `os.path.exists(DIR_OUTPUT)`。

条件成立才执行相应缩进分支，不成立则转到后续 elif/else 或跳过。

- 求值 `os.path.exists(DIR_OUTPUT)`：检查路径是否存在，返回布尔值。传入：位置参数1是 `DIR_OUTPUT`
- 求值 `not os.path.exists(DIR_OUTPUT)`：对条件取逻辑否定，作用对象是 `os.path.exists(DIR_OUTPUT)`

- 调用 `os.path.exists` 时的位置参数1：`DIR_OUTPUT`；先求出该表达式再传入

- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。

**第 4 行**

```python
    os.makedirs(DIR_OUTPUT)
```

执行位置：仅在条件 `not os.path.exists(DIR_OUTPUT)` 成立的分支中执行

创建目录；这会写入文件系统。传入：位置参数1是 `DIR_OUTPUT`。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `os.makedirs(DIR_OUTPUT)`：创建目录；这会写入文件系统。传入：位置参数1是 `DIR_OUTPUT`

- 调用 `os.makedirs` 时的位置参数1：`DIR_OUTPUT`；先求出该表达式再传入

- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 6 行**

```python
start_date = datetime.datetime.strptime("2018-04-01", "%Y-%m-%d")
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

先计算右侧：按格式字符串把文字日期解析为 datetime 对象。传入：位置参数1是 `'2018-04-01'`；位置参数2是 `'%Y-%m-%d'`。

把结果绑定到 `start_date`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `datetime.datetime.strptime('2018-04-01', '%Y-%m-%d')`：按格式字符串把文字日期解析为 datetime 对象。传入：位置参数1是 `'2018-04-01'`；位置参数2是 `'%Y-%m-%d'`

- 调用 `datetime.datetime.strptime` 时的位置参数1：`'2018-04-01'`；先求出该表达式再传入
- 调用 `datetime.datetime.strptime` 时的位置参数2：`'%Y-%m-%d'`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。

**第 8 行**

```python
for day in range(transactions_df.TX_TIME_DAYS.max()+1):
```

执行位置：处于单元格顶层；真正执行本cell并到达此语句时才发生，静态精读本身不执行代码

遍历 `range(transactions_df.TX_TIME_DAYS.max() + 1)`，每次把取出的项交给 `day`。

对每项执行缩进体；循环变量通常被下一项覆盖，累计结果需另放列表、表格或计数器。空可迭代对象不会执行循环体。

- 求值 `transactions_df.TX_TIME_DAYS.max()`：取最大值；DataFrame 分列取最大值时不同列可能来自不同原行。这里没有显式传入参数
- 求值 `transactions_df.TX_TIME_DAYS.max() + 1`：先计算 `transactions_df.TX_TIME_DAYS.max()` 与 `1`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `range(transactions_df.TX_TIME_DAYS.max() + 1)`：创建供循环使用的整数范围，终点不包含在内。传入：位置参数1是 `transactions_df.TX_TIME_DAYS.max() + 1`

- 调用 `range` 时的位置参数1：`transactions_df.TX_TIME_DAYS.max() + 1`；先求出该表达式再传入

- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 冒号在语句末尾引出缩进块，在方括号内可表示切片，在字典内分隔键和值。

- `TX_TIME_DAYS`：相对模拟起点的整天编号
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

**第 10 行**

```python
    transactions_day = transactions_df[transactions_df.TX_TIME_DAYS==day].sort_values('TX_TIME_SECONDS')
```

执行位置：属于循环体，仅在本轮从 `range(transactions_df.TX_TIME_DAYS.max() + 1)` 取得一项赋给 `day`后走到本行时执行

先计算右侧：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_TIME_SECONDS'`。

把结果绑定到 `transactions_day`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `transactions_df.TX_TIME_DAYS == day`：判断`transactions_df.TX_TIME_DAYS` 是否等于 `day`；标量比较通常给一个布尔值，数组或Series比较通常逐项给布尔值
- 求值 `transactions_df[transactions_df.TX_TIME_DAYS == day]`：从 `transactions_df` 中按 `transactions_df.TX_TIME_DAYS == day` 取值；字符串常指列或字典键，布尔序列常用于筛选，整数的含义取决于对象的索引规则
- 求值 `transactions_df[transactions_df.TX_TIME_DAYS == day].sort_values('TX_TIME_SECONDS')`：按给定列排序；默认返回排序结果，只有明确 inplace 才原地修改。传入：位置参数1是 `'TX_TIME_SECONDS'`

- 调用 `transactions_df[transactions_df.TX_TIME_DAYS == day].sort_values` 时的位置参数1：`'TX_TIME_SECONDS'`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 比较符生成判断结果；`==` 比较值是否相等，`=` 用于赋值或具名参数。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 方括号可能创建列表，也可能是已有对象的索引/切片。`表[列名]`、`表[条件]` 与 `[值1,值2]` 的用途不同。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

- `TX_TIME_DAYS`：相对模拟起点的整天编号
- `transactions_df`：交易明细 DataFrame，每行通常对应一笔交易

**第 12 行**

```python
    date = start_date + datetime.timedelta(days=day)
```

执行位置：属于循环体，仅在本轮从 `range(transactions_df.TX_TIME_DAYS.max() + 1)` 取得一项赋给 `day`后走到本行时执行

先计算右侧：先计算 `start_date` 与 `datetime.timedelta(days=day)`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算。

把结果绑定到 `date`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `datetime.timedelta(days=day)`：构造时间间隔，关键字 days/seconds 决定单位。传入：`days=day` 是具名参数
- 求值 `start_date + datetime.timedelta(days=day)`：先计算 `start_date` 与 `datetime.timedelta(days=day)`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算

- 调用 `datetime.timedelta` 时的具名参数 `days` 接收 `day`

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 13 行**

```python
    filename_output = date.strftime("%Y-%m-%d")+'.pkl'
```

执行位置：属于循环体，仅在本轮从 `range(transactions_df.TX_TIME_DAYS.max() + 1)` 取得一项赋给 `day`后走到本行时执行

先计算右侧：先计算 `date.strftime('%Y-%m-%d')` 与 `'.pkl'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算。

把结果绑定到 `filename_output`。名称原来的引用被替换；若右侧只是另一个对象变量，这通常是共享引用，不是自动复制。

- 求值 `date.strftime('%Y-%m-%d')`：按格式把日期对象转换成文字。传入：位置参数1是 `'%Y-%m-%d'`
- 求值 `date.strftime('%Y-%m-%d') + '.pkl'`：先计算 `date.strftime('%Y-%m-%d')` 与 `'.pkl'`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算

- 调用 `date.strftime` 时的位置参数1：`'%Y-%m-%d'`；先求出该表达式再传入

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 15 行**

```python
    # Protocol=4 required for Google Colab
```

执行位置：本行是注释，不执行。所在上下文：属于循环体，仅在本轮从 `range(transactions_df.TX_TIME_DAYS.max() + 1)` 取得一项赋给 `day`后走到本行时执行

原作者的说明性注释：Protocol=4 required for Google Colab

该行不执行赋值、计算或训练。注释与实际实现不一致时，需按代码核对。

- 本行 `#` 注释供人阅读，不参与运算；字符串内部的 # 不是注释标记。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

**第 16 行**

```python
    transactions_day.to_pickle(DIR_OUTPUT+filename_output, protocol=4)
```

执行位置：属于循环体，仅在本轮从 `range(transactions_df.TX_TIME_DAYS.max() + 1)` 取得一项赋给 `day`后走到本行时执行

把当前对象序列化到指定文件；这是实际写盘动作。传入：位置参数1是 `DIR_OUTPUT + filename_output`；`protocol=4` 是具名参数。

本行没有用等号把返回值另存到名称。若是对象方法可能有副作用；若是单元格最后一个表达式，Jupyter通常会显示其返回结果。

- 求值 `DIR_OUTPUT + filename_output`：先计算 `DIR_OUTPUT` 与 `filename_output`；数值的 + 是相加，列表或字符串的 + 是按顺序拼接，数组/Series则按各自对齐规则计算
- 求值 `transactions_day.to_pickle(DIR_OUTPUT + filename_output, protocol=4)`：把当前对象序列化到指定文件；这是实际写盘动作。传入：位置参数1是 `DIR_OUTPUT + filename_output`；`protocol=4` 是具名参数

- 调用 `transactions_day.to_pickle` 时的位置参数1：`DIR_OUTPUT + filename_output`；先求出该表达式再传入
- 调用 `transactions_day.to_pickle` 时的具名参数 `protocol` 接收 `4`

- `=` 在赋值位置把右侧结果交给左侧；在调用的 `参数名=值` 中则指定参数，不是在比较。
- 点号从左侧对象或模块中取属性/方法；后接 `(...)` 才是在调用可调用成员。
- 圆括号在可调用表达式后可表示调用，也可用于运算分组、元组或类的基类声明；应结合当前语句。
- 逗号分隔参数、元素或解包目标，项目的先后顺序可能决定字段或维度对应。
- 行首缩进说明它处于某个函数、循环、分支或括号续行中；不要仅看视觉对齐就判断执行次数。

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。

In [22]:
DIR_OUTPUT = "./simulated-data-transformed/"

if not os.path.exists(DIR_OUTPUT):
    os.makedirs(DIR_OUTPUT)

start_date = datetime.datetime.strptime("2018-04-01", "%Y-%m-%d")

for day in range(transactions_df.TX_TIME_DAYS.max()+1):
    
    transactions_day = transactions_df[transactions_df.TX_TIME_DAYS==day].sort_values('TX_TIME_SECONDS')
    
    date = start_date + datetime.timedelta(days=day)
    filename_output = date.strftime("%Y-%m-%d")+'.pkl'
    
    # Protocol=4 required for Google Colab
    transactions_day.to_pickle(DIR_OUTPUT+filename_output, protocol=4)

The generated dataset is also available from Github at `https://github.com/Fraud-Detection-Handbook/simulated-data-transformed`.

## 源码精读 · 原单元格 38

**静态逐行解析。** 原书代码未改；本文件未执行原代码。末尾已有输出继承自原书，不代表本次运行。

**这格解决什么：** 沿原笔记本顺序理解这格的输入、调用与结果。

**执行前：** 按前文准备好本格所依赖的库与对象；只阅读页面不会改变 Notebook 状态。

**执行后：** 是否产生新对象、改变原对象或只是显示，见每行“动作与状态变化”。本解析不推断未知运行数值。

下面逐行保留原代码，并说明右侧先计算什么、左侧接住什么，以及对象是否原地改变。若本格只是展示表或画图，应回到最近一次赋值寻找数据来源；原书保存的输出可在末尾展开核对。

### 按物理代码行精读（不含纯空行）

### 下面是原代码单元格

仅阅读无需执行。已有结果是原书历史输出。